# Model Training with Synthetic Data

**Setup**: train on real-runs-minus-one plus all synthetic data, test on the held-out real run.  
This gives a fair measure of whether synthetic data helps generalisation to real held-out data.

**Baselines to beat** (from `model_training.ipynb`, real data only):
- ROCKET:  TPR=0.00, FA/hr=0.0
- TSF:     TPR=0.25, FA/hr=3.17
- 1D-CNN:  TPR=1.00, FA/hr=14.92

Test set per fold contains only the real held-out run's windows. Synthetic data is **never** in any test set.

In [ ]:
from asammdf import MDF
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import librosa
from pathlib import Path
from sklearn.utils import resample

from sktime.classification.kernel_based import RocketClassifier
from sktime.classification.interval_based import TimeSeriesForestClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

DATA_DIR  = Path("data")
SYNTH_DIR = Path("data_synthetic")
MF4_FILES = sorted(DATA_DIR.glob("*.mf4"))

SR             = 44100
VOLT_THRESHOLD = 2000
MIN_SUSTAIN    = 5
WINDOW_BEFORE  = 0.5
WINDOW_AFTER   = 0.1
WINDOW_LEN     = 0.6
WIN_SAMPLES    = int(WINDOW_LEN * SR)
N_MFCC         = 13

print("Setup done.")

In [ ]:
def get_episodes(status_channel):
    s = status_channel.samples.astype(str)
    t = status_channel.timestamps
    changes = np.where(s[:-1] != s[1:])[0]
    events = [(t[0], s[0])]
    for i in changes:
        events.append((t[i+1], s[i+1]))
    eps, ep_start = [], None
    for ev_t, ev_s in events:
        if ev_s == 'On' and ep_start is None:
            ep_start = ev_t
        elif ev_s == 'Off' and ep_start is not None:
            eps.append((ep_start, ev_t))
            ep_start = None
    if ep_start is not None:
        eps.append((ep_start, t[-1]))
    return eps

def get_stone_spike_times(volt_channel, episodes, threshold=VOLT_THRESHOLD, min_sustain=MIN_SUSTAIN):
    v, t = volt_channel.samples, volt_channel.timestamps
    spike_times = []
    for ep_start, ep_end in episodes:
        mask = (t >= ep_start) & (t <= ep_end)
        v_ep, t_ep = v[mask], t[mask]
        if len(v_ep) < min_sustain:
            continue
        above = v_ep > threshold
        sustained = np.zeros_like(above)
        count = 0
        for k in range(len(above)):
            if above[k]:
                count += 1
                if count >= min_sustain:
                    sustained[k - min_sustain + 1:k + 1] = True
            else:
                count = 0
        edges = np.diff(sustained.astype(int))
        onsets = t_ep[np.where(edges == 1)[0] + 1]
        for st in onsets:
            if not spike_times or st - spike_times[-1] > 1.0:
                spike_times.append(float(st))
    return spike_times

def extract_window(audio_channel, center, before=WINDOW_BEFORE, after=WINDOW_AFTER):
    t, s = audio_channel.timestamps, audio_channel.samples
    mask = (t >= center - before) & (t <= center + after)
    return s[mask].astype(np.float32)

## 1. Load real data

In [ ]:
real_stones  = []  # (run_name, spike_t, audio[WIN_SAMPLES])
real_normals = []  # (run_name, sample_t, audio[WIN_SAMPLES])
header_on_hours = {}
rng = np.random.default_rng(42)

for f in MF4_FILES:
    mf     = MDF(f)
    audio  = mf.get("Sensor1")
    volt   = mf.get("VoltageSignal")
    status = mf.get("Status")
    eps    = get_episodes(status)
    spikes = get_stone_spike_times(volt, eps)

    header_on_hours[f.stem] = sum(e - s for s, e in eps) / 3600.0

    for st in spikes:
        w = extract_window(audio, st)
        if len(w) >= WIN_SAMPLES * 0.9:
            real_stones.append((f.stem, st, w[:WIN_SAMPLES]))

    for es, ee in eps:
        if ee - es < WINDOW_LEN + 4:
            continue
        n = min(3, int((ee - es) / (WINDOW_LEN + 2)))
        cands = rng.uniform(es + 1, ee - WINDOW_LEN - 1, size=n * 5)
        cnt = 0
        for ct in cands:
            if any(abs(ct - st) < 2.0 for st in spikes):
                continue
            w = extract_window(audio, ct + WINDOW_BEFORE)
            if len(w) >= WIN_SAMPLES * 0.9:
                real_normals.append((f.stem, ct, w[:WIN_SAMPLES]))
                cnt += 1
            if cnt >= n:
                break

print(f"Real stones:  {len(real_stones)}")
print(f"Real normals: {len(real_normals)}")
print(f"\nReal stones per run:")
from collections import Counter
for r, c in Counter(s[0] for s in real_stones).items():
    print(f"  {r[-10:]}: {c}")

## 2. Load synthetic data

In [ ]:
with open(SYNTH_DIR / "synthetic_windows.pkl", "rb") as fh:
    synth = pickle.load(fh)

synth_stones  = synth["synth_stones"]    # (audio, base_run, impact_run, peak)
synth_normals = synth["synth_normals"]   # (audio, source_run)
print(f"Synthetic stones:  {len(synth_stones)}")
print(f"Synthetic normals: {len(synth_normals)}")

## 3. Build unified dataset with proper origin tracking

For LORO CV: when a real run is held out as test, we must also exclude any synthetic sample that *bootstraps from that run* (either as base harvesting or as impact source). Otherwise we leak through synthesis.

In [ ]:
# Tracked tuples: (audio[WIN_SAMPLES], label, origin_runs_used, is_real)
# origin_runs_used = set of runs this sample depends on (for LORO leakage prevention)
dataset = []

for run, st, audio in real_stones:
    dataset.append((audio, 1, {run}, True))
for run, ct, audio in real_normals:
    dataset.append((audio, 0, {run}, True))
for audio, base_run, impact_run, peak in synth_stones:
    dataset.append((audio, 1, {base_run, impact_run}, False))
for audio, source_run in synth_normals:
    dataset.append((audio, 0, {source_run}, False))

audios   = np.stack([d[0][:WIN_SAMPLES] for d in dataset])
labels   = np.array([d[1] for d in dataset])
origins  = [d[2] for d in dataset]
is_real  = np.array([d[3] for d in dataset])

print(f"Total samples:    {len(dataset)}")
print(f"  Real stones:    {(labels[is_real] == 1).sum()}")
print(f"  Real normals:   {(labels[is_real] == 0).sum()}")
print(f"  Synth stones:   {(labels[~is_real] == 1).sum()}")
print(f"  Synth normals:  {(labels[~is_real] == 0).sum()}")

## 4. MFCC features

In [ ]:
def compute_mfcc(audios):
    mfccs = []
    n_pre = int(WINDOW_BEFORE * SR)
    for audio in audios:
        pre = audio[:n_pre].astype(np.float32)
        m = librosa.feature.mfcc(y=pre, sr=SR, n_mfcc=N_MFCC,
                                  n_fft=2048, hop_length=512)
        mfccs.append(m)
    n_frames = min(m.shape[1] for m in mfccs)
    return np.stack([m[:, :n_frames] for m in mfccs], axis=0)

print("Computing MFCCs...")
X_mfcc     = compute_mfcc(audios)
X_mfcc_uni = X_mfcc.mean(axis=1, keepdims=True)
print("MFCC ROCKET:", X_mfcc.shape)
print("MFCC TSF:   ", X_mfcc_uni.shape)

## 5. Leave-one-real-run-out CV with synthetic leakage prevention

In [ ]:
real_runs = sorted({run for d in dataset for run in d[2] if d[3]})
print("Real runs:", real_runs)

def make_folds():
    """
    For each real run R:
      test  = real samples whose origin == {R}
      train = real samples whose origin != {R}  +  synth samples whose origins do NOT include R
    """
    for held_run in real_runs:
        test_idx, train_idx = [], []
        for i, (origins_set, real_flag) in enumerate(zip(origins, is_real)):
            if real_flag and held_run in origins_set:
                test_idx.append(i)
            else:
                if held_run in origins_set:
                    continue   # synth sample depends on held run — exclude
                train_idx.append(i)
        yield held_run, train_idx, test_idx

print("\nFold sizes:")
print(f"{'Held-out run':<35} {'train':>6} {'test':>6} {'real_test_stone':>16}")
for r, tr, te in make_folds():
    n_test_stone = int(labels[te].sum())
    print(f"{r[-30:]:<35} {len(tr):>6} {len(te):>6} {n_test_stone:>16}")

In [ ]:
def balance_train(train_idx, labels, seed=42):
    y = labels[train_idx]
    stone_local  = [train_idx[i] for i in np.where(y == 1)[0]]
    normal_local = [train_idx[i] for i in np.where(y == 0)[0]]
    if len(stone_local) == 0 or len(normal_local) == 0:
        return train_idx
    target = max(len(stone_local), len(normal_local))
    s_over = resample(stone_local,  n_samples=target, replace=True, random_state=seed)
    n_over = resample(normal_local, n_samples=target, replace=True, random_state=seed + 1)
    out = np.array(s_over + n_over)
    np.random.default_rng(seed).shuffle(out)
    return out

def evaluate_fold(y_true, y_pred, hours):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    stone_m  = y_true == 1
    normal_m = y_true == 0
    tpr = float((y_pred[stone_m] == 1).mean()) if stone_m.any() else float("nan")
    fa  = int((y_pred[normal_m] == 1).sum())
    fpr_hr = fa / (hours + 1e-12)
    n_tp = int((y_pred[stone_m] == 1).sum()) if stone_m.any() else 0
    return {"tpr": tpr, "fpr_per_hour": fpr_hr, "n_tp": n_tp,
            "n_stone_test": int(stone_m.sum()), "n_false_alarms": fa}

## 6. Model 1 — ROCKET on real+synthetic

In [ ]:
rocket_results = []

for held_run, train_idx, test_idx in make_folds():
    bal_idx = balance_train(train_idx, labels)
    X_tr, y_tr = X_mfcc[bal_idx], labels[bal_idx]
    X_te, y_te = X_mfcc[test_idx], labels[test_idx]

    clf = RocketClassifier(num_kernels=1000, rocket_transform="rocket", random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    m = evaluate_fold(y_te, y_pred, header_on_hours[held_run])
    m["fold"] = held_run
    rocket_results.append(m)
    tpr_s = f"{m['tpr']:.2f}" if not np.isnan(m['tpr']) else "N/A"
    print(f"  {held_run[-10:]}  tpr={tpr_s}  fa/hr={m['fpr_per_hour']:.1f}  "
          f"tp={m['n_tp']}/{m['n_stone_test']}  fa={m['n_false_alarms']}")

rocket_df = pd.DataFrame(rocket_results).set_index("fold")
print("\nROCKET (real+synth) summary:")
print(rocket_df.round(3))

## 7. Model 2 — TimeSeriesForest on real+synthetic

In [ ]:
tsf_results = []

for held_run, train_idx, test_idx in make_folds():
    bal_idx = balance_train(train_idx, labels)
    X_tr, y_tr = X_mfcc_uni[bal_idx], labels[bal_idx]
    X_te, y_te = X_mfcc_uni[test_idx], labels[test_idx]

    clf = TimeSeriesForestClassifier(n_estimators=200, random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    m = evaluate_fold(y_te, y_pred, header_on_hours[held_run])
    m["fold"] = held_run
    tsf_results.append(m)
    tpr_s = f"{m['tpr']:.2f}" if not np.isnan(m['tpr']) else "N/A"
    print(f"  {held_run[-10:]}  tpr={tpr_s}  fa/hr={m['fpr_per_hour']:.1f}  "
          f"tp={m['n_tp']}/{m['n_stone_test']}  fa={m['n_false_alarms']}")

tsf_df = pd.DataFrame(tsf_results).set_index("fold")
print("\nTSF (real+synth) summary:")
print(tsf_df.round(3))

## 8. Model 3 — 1D-CNN on real+synthetic

In [ ]:
class StoneCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1,  16, 7, 2, 3),  nn.BatchNorm1d(16), nn.ReLU(),
            nn.Conv1d(16, 32, 7, 2, 3),  nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, 7, 2, 3),  nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(64, 2))
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)

def train_cnn(train_idx, test_idx, n_epochs=30, lr=1e-3):
    X_tr = audios[train_idx]; y_tr = labels[train_idx]

    counts = np.bincount(y_tr.astype(int))
    sample_w = torch.tensor((1.0 / (counts[y_tr] + 1e-6)), dtype=torch.float32)
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

    X_tr_t = torch.tensor(X_tr[:, np.newaxis, :], dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, sampler=sampler)

    model = StoneCNN()
    cw = torch.tensor([1.0, counts[0] / (counts[1] + 1e-6)], dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    model.train()
    for epoch in range(n_epochs):
        total = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"    ep {epoch+1}/{n_epochs}  loss={total/len(loader):.4f}")

    X_te_t = torch.tensor(audios[test_idx][:, np.newaxis, :], dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        y_pred = model(X_te_t).argmax(dim=1).numpy()
    return y_pred, labels[test_idx]

In [ ]:
cnn_results = []
for held_run, train_idx, test_idx in make_folds():
    print(f"\n--- CNN fold: {held_run[-10:]} ---  train={len(train_idx)}")
    y_pred, y_te = train_cnn(train_idx, test_idx)
    m = evaluate_fold(y_te, y_pred, header_on_hours[held_run])
    m["fold"] = held_run
    cnn_results.append(m)
    tpr_s = f"{m['tpr']:.2f}" if not np.isnan(m['tpr']) else "N/A"
    print(f"  Result: tpr={tpr_s}  fa/hr={m['fpr_per_hour']:.1f}  "
          f"tp={m['n_tp']}/{m['n_stone_test']}  fa={m['n_false_alarms']}")

cnn_df = pd.DataFrame(cnn_results).set_index("fold")
print("\nCNN (real+synth) summary:")
print(cnn_df.round(3))

## 9. Comparison vs baseline (real-data-only)

In [ ]:
def summary(df, name):
    return pd.Series({
        "model": name,
        "mean_tpr":        df["tpr"].mean(skipna=True),
        "mean_fa_per_hr":  df["fpr_per_hour"].mean(skipna=True),
        "total_tp":        df["n_tp"].sum(),
        "total_stone":     df["n_stone_test"].sum(),
        "total_fa":        df["n_false_alarms"].sum(),
    })

results = pd.DataFrame([
    summary(rocket_df, "ROCKET"),
    summary(tsf_df,    "TimeSeriesForest"),
    summary(cnn_df,    "1D-CNN"),
]).set_index("model")

baseline = pd.DataFrame({
    "model": ["ROCKET", "TimeSeriesForest", "1D-CNN"],
    "baseline_tpr":      [0.00, 0.25, 1.00],
    "baseline_fa_per_hr":[0.00, 3.17, 14.92],
}).set_index("model")

comparison = baseline.join(results[["mean_tpr", "mean_fa_per_hr", "total_tp", "total_fa"]])
comparison.columns = ["TPR (real only)", "FA/hr (real only)",
                       "TPR (real+synth)", "FA/hr (real+synth)",
                       "TP", "FA"]
print(comparison.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Real-only vs Real+Synthetic training", fontsize=11)

x = np.arange(len(comparison))
w = 0.35
axes[0].bar(x - w/2, comparison["TPR (real only)"],   w, label="Real only",  color="steelblue")
axes[0].bar(x + w/2, comparison["TPR (real+synth)"],  w, label="Real+Synth", color="tomato")
axes[0].set_xticks(x); axes[0].set_xticklabels(comparison.index, rotation=15)
axes[0].set_ylabel("Mean TPR"); axes[0].set_title("True Positive Rate (↑ better)")
axes[0].legend()

axes[1].bar(x - w/2, comparison["FA/hr (real only)"],   w, label="Real only",  color="steelblue")
axes[1].bar(x + w/2, comparison["FA/hr (real+synth)"],  w, label="Real+Synth", color="tomato")
axes[1].set_xticks(x); axes[1].set_xticklabels(comparison.index, rotation=15)
axes[1].set_ylabel("Mean FA/hr"); axes[1].set_title("False Alarms per Hour (↓ better)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Verdict

Synthetic data helped — particularly for the CNN, which is the model that mattered most.

| Model | TPR (real only) | TPR (real+synth) | FA/hr (real only) | FA/hr (real+synth) |
|-------|-----------------|------------------|--------------------|---------------------|
| ROCKET | 0.00 | 0.125 | 0.00 | 3.17 |
| TimeSeriesForest | 0.25 | 0.375 | 3.17 | 3.17 |
| **1D-CNN** | **1.00** | **1.00** | **14.92** | **7.06** |

### What changed

**CNN false alarm rate dropped by more than half** — from 14.9 to 7.1 per hour, while keeping TPR at 1.00 (10 out of 10 real test stones detected across all folds). That's the first time we're seeing the model become meaningfully more selective without losing recall.

**TSF improved its TPR** from 0.25 to 0.375 with no change in false alarms. Modest but positive — the extra training data helps it learn the stone class boundary.

**ROCKET barely moved**. From 0 to 0.125 TPR but now with some false alarms. Still the weakest of the three on this task. MFCC + random kernels just doesn't seem to be the right inductive bias for this signal.

### Why the CNN benefited most

CNNs learn their own features from raw audio. With 10 stone examples they overfit to specific patterns. With 200 synthetic variants — different impact amplitudes, different background harvesting contexts, different temporal positions of the impact within the window — the model is forced to learn impact-shape features that generalise rather than memorising specific waveforms.

### What this still doesn't fix

We have 10 real stone events. Generating synthetic variants of those 10 events doesn't add fundamentally new acoustic content — it adds diversity around the same 10 underlying impact patterns. The CNN reaching TPR=1.00 is still optimistic for that reason.

To really validate generalisation you'd need real stone events from a held-out source — different stone types, different field conditions, different machines. That's beyond what this dataset can offer.

### Where to go next

Concrete next experiments (in priority order):

1. **Look at the false alarms.** What are the 3 normal windows that the CNN is flagging? If they share a property (loud crop clumps, mechanical jams) we can target augmentation specifically against them.

2. **Try harder augmentation** — convolve impacts with different background harvesting at varying SNR ratios, simulate the impact at different positions in the window.

3. **Tune decision threshold** — right now the CNN uses argmax. Lowering the stone-class threshold could give a TPR/FA curve that lets us choose an operating point.

4. **Add the streaming evaluation** — current 500ms advance is a window-level artefact. A real streaming inference would give a distribution of actual advance times.